# 10 - Lalonde / NSW benchmark

This notebook uses the public LaLonde-style benchmark for a realistic observational causal workflow.


## Causal question

What is the average effect of treatment assignment on post-training earnings, and how do methods compare when evaluated on the same covariate set?

Source: [Rdatasets MatchIt Lalonde CSV](https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/MatchIt/lalonde.csv).
This dataset is prepared locally into `data/processed/lalonde_job_training.csv` by script in `scripts/`.

## Identification assumptions

- Ignorability conditional on observed covariates (unconfoundedness).
- No interference between units.
- Sufficient overlap in treatment probabilities.

## Estimand

Average treatment effect on the treated population from this benchmark sample (ATT proxy), compared across methods.

## Unit of analysis

Each row represents one participant in the job training dataset.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
SCRIPTS_PATH = PROJECT_ROOT / "scripts"
for path in (SRC_PATH, SCRIPTS_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import pandas as pd

from causal_inference_lab.estimators import aipw_ate, difference_in_means, g_computation_ate, ipw_ate
from causal_inference_lab.matching import nearest_neighbour_matching
from causal_inference_lab.diagnostics import balance_table
from causal_inference_lab.sensitivity import omitted_confounder_simulation, placebo_treatment_test

from prepare_lalonde_job_training_dataset import prepare_lalonde_job_training_dataset

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "lalonde_job_training.csv"
if not DATA_PATH.exists():
    prepare_lalonde_job_training_dataset()

data = pd.read_csv(DATA_PATH)
covariates = ["age", "education", "black", "hispanic", "married", "nodegree", "earnings_74", "earnings_75", "u74", "u75"]


In [ ]:
naive = difference_in_means(data, treatment_col="treatment", outcome_col="outcome")
ipw = ipw_ate(data, covariates, treatment_col="treatment", outcome_col="outcome")
gcomp = g_computation_ate(data, covariates, treatment_col="treatment", outcome_col="outcome")
aipw = aipw_ate(data, covariates, treatment_col="treatment", outcome_col="outcome")
matching = nearest_neighbour_matching(data, covariates, treatment_col="treatment", outcome_col="outcome")

print(f"Naive diff-in-means: {naive.estimate:.3f}")
print(f"IPW:                {ipw.estimate:.3f}")
print(f"g-computation:      {gcomp.estimate:.3f}")
print(f"AIPW:               {aipw.estimate:.3f}")
print(f"Nearest-neighbor ATT: {matching.effect.estimate:.3f}")


## Diagnostics, uncertainty, and limitation checks

The same benchmark should be stress-tested before any causal narrative.


In [ ]:
before = balance_table(data, covariates=covariates, treatment_col="treatment")
print(f"Average absolute SMD before weighting: {before['abs_smd'].mean():.3f}")

placebo = placebo_treatment_test(
    data=data,
    estimator=lambda frame, cols: aipw_ate(frame, cols, treatment_col="treatment", outcome_col="outcome"),
    covariates=covariates,
    treatment_col="treatment",
    seed=42,
)
print(f"Placebo AIPW: {placebo.estimate:.3f}")

sensitivity = omitted_confounder_simulation(
    data=data,
    base_effect=aipw.estimate,
    confounder_strength_grid=[0.0, 0.2, 0.4, 0.6, 0.8],
    treatment_col="treatment",
)
print(sensitivity[["confounder_strength", "adjusted_effect"]])


## Interpretation

No single number is a universal truth. Methods here are sensitive to overlap and modeling assumptions, and benchmark performance depends heavily on feature specification and balance diagnostics.
